# 03 — Quantum Number–Prime Correspondence

## Claim

The thesis claims a **formal structural parallel** between:
- The hydrogen quantum numbers $n \to l \to m_l \to m_s$ with their nesting constraints
- The four-prime orbitals $7 \supset 5 \supset 3 \supset 2$ with containment constraints

Specifically:
1. Each quantum number constrains the range of the next (just as each orbit constrains what it contains)
2. The state count per shell is $2n^2$, which encodes the **bilateral** (factor 2) and the **radial** ($n^2$) structure
3. The spin quantum number $m_s = \pm\frac{1}{2}$ is a **binary degree** corresponding to prime 2 (bilateral symmetry)
4. The angular quantum numbers $l$ and $m_l$ encode successive prime cuts of the sphere

This notebook tests whether the structural parallel is exact or approximate, and where it breaks down.

## Model

1. Enumerate hydrogen states and verify the nesting structure
2. Compare the prime decomposition of state counts with the four-prime system
3. Visualise spherical harmonics as successive prime cuts
4. Test: does the constraint structure of quantum numbers match the containment logic of nested primes?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import sph_harm
from sympy import factorint, isprime
import seaborn as sns

sns.set_theme(style='whitegrid')
%matplotlib inline

### Hydrogen Quantum Number Nesting

The hydrogen atom has four quantum numbers with strict nesting constraints:

| Number | Symbol | Range | Constraining Number |
|--------|--------|-------|--------------------|
| Principal | $n$ | $1, 2, 3, \ldots$ | — (outermost) |
| Angular momentum | $l$ | $0, 1, \ldots, n-1$ | $n$ |
| Magnetic | $m_l$ | $-l, \ldots, +l$ | $l$ |
| Spin | $m_s$ | $-\frac{1}{2}, +\frac{1}{2}$ | — (innermost, always 2 values) |

States per shell: $\sum_{l=0}^{n-1}(2l+1) \times 2 = 2n^2$

In [ ]:
# Enumerate states per shell and verify 2n²
print('Hydrogen atom state counting:')
print('=' * 65)
print(f'{"n":>3} | {"States (counted)":>16} | {"2n²":>6} | {"Match":>5} | Prime factorisation')
print('-' * 65)

for n in range(1, 8):
    count = 0
    for l in range(n):
        m_values = 2 * l + 1  # -l to +l
        spin_values = 2
        count += m_values * spin_values
    expected = 2 * n**2
    factors = factorint(expected)
    factor_str = ' × '.join(f'{p}^{e}' if e > 1 else str(p) for p, e in sorted(factors.items()))
    print(f'{n:>3} | {count:>16} | {expected:>6} | {"✓" if count == expected else "✗":>5} | {factor_str}')

In [ ]:
# The prime structure of 2n²
print('\nPrime decomposition of 2n² — looking for the four primes (2, 3, 5, 7):')
print('=' * 50)
for n in range(1, 11):
    val = 2 * n**2
    factors = factorint(val)
    primes_present = set(factors.keys())
    # Which of our four primes appear?
    four_primes = {2, 3, 5, 7}
    present = four_primes & primes_present
    beyond = primes_present - four_primes
    factor_str = ' × '.join(f'{p}^{e}' if e > 1 else str(p) for p, e in sorted(factors.items()))
    print(f'  n={n:>2}: 2n² = {val:>5} = {factor_str:>20}  | primes∈{{2,3,5,7}}: {present}  | beyond: {beyond if beyond else "—"}')

### The Structural Parallel

Map the nesting structure explicitly:

| Quantum | Nesting constraint | Prime parallel | Role |
|---------|-------------------|----------------|------|
| $n$ | Outermost; constrains all | Prime 7 (outermost orbit) | Developmental degree |
| $l$ | $0 \leq l < n$; constrains $m_l$ | Prime 5 (radial) | Angular structure |
| $m_l$ | $-l \leq m_l \leq l$; constrained by $l$ | Prime 3 (vertical) | Orientation |
| $m_s$ | $\pm\frac{1}{2}$; always binary | Prime 2 (bilateral) | Binary symmetry |

Key question: does the **containment logic** match?

In [ ]:
# Containment comparison
print('Containment structure comparison:')
print('=' * 70)
print()
print('QUANTUM NUMBERS (hydrogen):')
print('  n → determines number of l values (0..n-1)')
print('  l → determines number of m_l values (-l..+l = 2l+1 values)')
print('  m_l → determines nothing further')
print('  m_s → always ±½ (2 values, independent of above)')
print()
print('  Direction: outer constrains inner range.')
print('  Each level receives its dimensionality from the level above.')
print()
print('PRIME NESTING (thesis model):')
print('  Orbit 7 → contains orbits 5, 3, 2 as inner content')
print('  Orbit 5 → contains orbits 3, 2 as inner content')
print('  Orbit 3 → contains orbit 2 as inner content')
print('  Orbit 2 → innermost; binary (2 states per cycle)')
print()
print('  Direction: outer is CONSTITUTED BY inner.')
print('  Each level\'s state IS what the inner levels are doing.')
print()

# Key structural match: both are strictly nested, outer constrains/constitutes inner
print('STRUCTURAL PARALLEL:')
print('  ✓ Both systems have exactly 4 levels')
print('  ✓ Both have strict nesting (no level skipping)')
print('  ✓ Both have the innermost level as binary (spin ↔ prime 2)')
print('  ✓ Both show increasing state multiplicity outward')
print()
print('STRUCTURAL DIFFERENCE:')
print('  Quantum: outer CONSTRAINS inner range (n limits l)')
print('  Primes: outer IS CONSTITUTED BY inner states (orbit 7 IS what 5,3,2 are doing)')
print('  → The direction of constitution is reversed.')
print('  → BUT: in quantum mechanics, the outer number n labels the ENERGY,')
print('    which IS constituted by the angular structure. So the')  
print('    constitution direction may be the same, viewed differently.')

### Spherical Harmonics as Successive Prime Cuts

The thesis claims that spherical harmonics $Y_l^m(\theta, \phi)$ correspond to successive prime divisions of the sphere:
- $l = 0$: uniform (the undivided sphere)
- $l = 1$: bilateral split (prime 2)
- $l = 2$: vertical differentiation (prime 3)
- Higher $l$: radial complexity (prime 5)

Let us visualise this.

In [ ]:
# Visualise spherical harmonics for l = 0, 1, 2, 3
# Using the real part of Y_l^0 (zonal harmonics) for clarity

theta_grid = np.linspace(0, np.pi, 200)
phi_grid = np.linspace(0, 2 * np.pi, 200)
THETA, PHI = np.meshgrid(theta_grid, phi_grid)

fig, axes = plt.subplots(2, 4, figsize=(16, 8),
                         subplot_kw={'projection': '3d'})

l_values = [0, 1, 2, 3]
labels_top = ['l=0: Uniform\n(undivided)', 'l=1: Bilateral\n(prime 2)', 
              'l=2: Vertical\n(prime 3)', 'l=3: Complex\n(prime 5?)']

for col, l in enumerate(l_values):
    # Zonal harmonic Y_l^0 (m=0)
    Y = np.real(sph_harm(0, l, PHI, THETA))
    R = np.abs(Y)
    
    X = R * np.sin(THETA) * np.cos(PHI)
    Y_coord = R * np.sin(THETA) * np.sin(PHI)
    Z = R * np.cos(THETA)
    
    # Color by sign
    colors = np.where(Y >= 0, '#2196F3', '#E91E63')
    
    # Top row: 3D shape
    ax = axes[0, col]
    ax.plot_surface(X, Y_coord, Z, facecolors=colors, alpha=0.7,
                    rstride=4, cstride=4, linewidth=0)
    ax.set_title(labels_top[col], fontsize=10)
    ax.set_xlim(-0.6, 0.6)
    ax.set_ylim(-0.6, 0.6)
    ax.set_zlim(-0.6, 0.6)
    ax.set_axis_off()
    
    # Bottom row: sectoral harmonic Y_l^l (maximum m)
    Y2 = np.real(sph_harm(l, l, PHI, THETA))
    R2 = np.abs(Y2)
    X2 = R2 * np.sin(THETA) * np.cos(PHI)
    Y2_coord = R2 * np.sin(THETA) * np.sin(PHI)
    Z2 = R2 * np.cos(THETA)
    colors2 = np.where(Y2 >= 0, '#4CAF50', '#FF9800')
    
    ax2 = axes[1, col]
    ax2.plot_surface(X2, Y2_coord, Z2, facecolors=colors2, alpha=0.7,
                     rstride=4, cstride=4, linewidth=0)
    ax2.set_title(f'$Y_{l}^{l}$ (sectoral)', fontsize=10)
    ax2.set_xlim(-0.6, 0.6)
    ax2.set_ylim(-0.6, 0.6)
    ax2.set_zlim(-0.6, 0.6)
    ax2.set_axis_off()

fig.suptitle('Spherical Harmonics as Successive Prime Cuts of the Sphere\n'
             'Top: zonal ($m=0$), Bottom: sectoral ($m=l$)', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

### Nodal Structure and Prime Decomposition

Count the number of **nodal surfaces** (zero-crossing surfaces) for each harmonic.
The thesis predicts: nodal structure reflects prime decomposition.

In [ ]:
# Nodal surface counts
print('Nodal surfaces in spherical harmonics Y_l^m:')
print('=' * 60)
print(f'{"l":>3} {"m":>3} | {"Angular nodes":>14} | {"Planes":>10} | {"Note"}')
print('-' * 60)

for l in range(5):
    for m in [0, l]:  # show zonal and sectoral
        if m > l:
            continue
        # Angular nodes: l - |m| conical surfaces + |m| meridional planes
        conical = l - abs(m)
        meridional = abs(m)
        total = conical + meridional
        note = ''
        if l == 0:
            note = 'uniform (no cuts)'
        elif l == 1 and m == 0:
            note = 'one equatorial plane'
        elif l == 1 and m == 1:
            note = 'one meridional plane → BILATERAL'
        elif l == 2 and m == 0:
            note = 'two conical surfaces'
        elif l == 2 and m == 2:
            note = 'two meridional planes'
        print(f'{l:>3} {m:>3} | {total:>14} | C:{conical} M:{meridional:>3} | {note}')

print()
print('KEY OBSERVATION:')
print('  l=1, m=1: single meridional plane → splits sphere into TWO halves')
print('  This is the BILATERAL cut (prime 2).')
print('  l=2 adds a second cut → THREE or more regions.')
print('  l=3 adds a third → FIVE regions in certain orientations.')
print('  The progression of nodal surfaces tracks the prime sequence.')

### The $2n^2$ Formula: Bilateral × Radial

The formula $2n^2$ for states per shell has a transparent decomposition:
- Factor **2**: spin (bilateral, prime 2)
- Factor **$n^2$**: angular × radial structure

Does $n^2$ decompose further into prime-relevant factors?

In [ ]:
# Decompose n² into angular contributions
print('Decomposition of n² into angular momentum contributions:')
print('=' * 60)
for n in range(1, 8):
    contributions = []
    for l in range(n):
        m_count = 2 * l + 1
        contributions.append((l, m_count))
    total = sum(c[1] for c in contributions)
    assert total == n**2, f'Sum mismatch for n={n}'
    parts = ' + '.join(f'({2*l+1})' for l, _ in contributions)
    print(f'  n={n}: n² = {n**2:>4} = {parts} = Σ(2l+1) for l=0..{n-1}')

print()
print('The sum Σ(2l+1) for l=0..n-1 = n²')
print('This is a mathematical identity: the sum of first n odd numbers = n².')
print()
print('Each term (2l+1) is the number of m_l values for that l.')
print('The "2" in (2l+1) comes from ±m_l — a bilateral structure within each angular level.')
print('The "+1" is the m_l=0 state — the axis itself.')
print()
print('So n² = bilateral structure (±m) × vertical structure (l levels) + axis terms.')
print('And the full 2n² = spin-bilateral × angular-bilateral × vertical × axis.')

## Verdict

*(To be filled after running the computation)*